In [1]:
import pandas as pd
import numpy as np

In [6]:
file_path = r'C:\Users\leeji\Desktop\study_csv\Informatics.csv'

try:
  # 한글 인코딩(utf-8 또는 cp949) 자동 대응
  try:
    df = pd.read_csv(file_path, encoding='utf-8')
  except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')
  print(f"'{file_path}' 인코딩 성공")
except FileNotFoundError:
  print(f"⚠️ '{file_path}' 인코딩 실패")

'C:\Users\leeji\Desktop\study_csv\Informatics.csv' 인코딩 성공


In [7]:
df.columns = df.columns.str.strip()

In [9]:
df['가티오_증감인원'] = df['2027 사전예고'] - df['2026 사전 예고']

df['가티오_증감률(%)'] = np.where(
    df['2026 사전 예고'] != 0,
    ((df['2027 사전예고'] - df['2026 사전 예고']) / df['2026 사전 예고']) * 100,
    0,
).round(2)

In [14]:
df['2026_가티오대비_본티오배수'] = np.where(
    df['2026 사전 예고'] != 0, df['2026 최종 일반'] / df['2026 사전 예고'], 1.0
)

df['2027_예측_최종선발'] = (
    (df['2027 사전예고'] * df['2026_가티오대비_본티오배수'])
    .round()
    .astype(int)
)

In [15]:
total_2027_pre = df['2027 사전예고'].sum()
total_2026_pre = df['2026 사전 예고'].sum()
total_2026_final = df['2026 최종 일반'].sum()
total_diff = total_2027_pre - total_2026_pre
total_rate = (
    round((total_diff / total_2026_pre) * 100, 2) if total_2026_pre != 0 else 0
)
total_ratio = (
    round(total_2026_final / total_2026_pre, 2) if total_2026_pre != 0 else 1.0
)

total_row = pd.DataFrame([{
    '지역': '합계',
    '2026 사전 예고': total_2026_pre,
    '2027 사전예고': total_2027_pre,
    '가티오_증감인원': total_diff,
    '가티오_증감률(%)': total_rate,
    '2026 최종 일반': total_2026_final,
    '2026_가티오대비_본티오배수': total_ratio,
    '2027_예측_최종선발': df['2027_예측_최종선발(배수모델)'].sum(),
}])

df_result = pd.concat([df, total_row], ignore_index=True)

display(
    df_result[[
        '지역',
        '2026 사전 예고',
        '2027 사전예고',
        '가티오_증감인원',
        '가티오_증감률(%)',
        '2026 최종 일반',
        '2026_가티오대비_본티오배수',
        '2027_예측_최종선발',
    ]]
)

,지역,2026 사전 예고,2027 사전예고,가티오_증감인원,가티오_증감률(%),2026 최종 일반,2026_가티오대비_본티오배수,2027_예측_최종선발
0,서울,19,25,6,31.58,23,1.210526,30
1,경기,79,36,-43,-54.43,115,1.455696,52
2,인천,20,31,11,55.00,20,1.000000,31
3,부산,20,17,-3,-15.00,20,1.000000,17
4,대구,7,8,1,14.29,9,1.285714,10
5,광주,1,3,2,200.00,3,3.000000,9
6,대전,2,4,2,100.00,4,2.000000,8
7,울산,6,8,2,33.33,10,1.666667,13
8,세종,5,9,4,80.00,5,1.000000,9
9,강원,5,8,3,60.00,9,1.800000,14
